In [1]:
import os, yaml, shutil, random, torch, json, wandb, tempfile, zipfile, glob
from pathlib import Path
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from collections import defaultdict
from huggingface_hub import HfApi, create_repo, upload_file, upload_folder
from datetime import datetime
from dotenv import load_dotenv
from PIL import Image

load_dotenv()  # .env 파일의 환경변수들을 로드

C:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
print("CUDA available:", torch.cuda.is_available())
print("torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("GPU 이름:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


CUDA available: True
torch version: 2.7.1+cu118
CUDA version: 11.8
cuDNN version: 90100
GPU 이름: NVIDIA GeForce RTX 3060


In [8]:
def setup_wandb(project_name="parking_gaurd", run_name=None):
    """
    Weights & Biases 초기화
    
    Args:
        project_name (str): wandb 프로젝트 이름
        run_name (str): 실행 이름 (None이면 자동 생성)
    """
    try:
        wandb.init(
            project=project_name,
            name=run_name,
            config={
                "model": "YOLOv8-seg",
                "classes": ["solid_yellow_lane", "dotted_yellow_lane", "double_yellow_lane", 
                           "crosswalk", "sidewalk", "firehydrant", "car", "license_plate"],
                "task": "instance_segmentation"
            }
        )
        print("wandb 초기화 완료")
        return True
    except Exception as e:
        print(f"wandb 초기화 실패: {e}")
        print("wandb 없이 계속 진행합니다...")
        return False

def split_dataset(source_images_dir, changed_labels_dir, output_dir, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):
    """
    train/val/test로 분할
    Args:
        source_images_dir (str): 원본 이미지 폴더 경로
        changed_labels_dir (str): 원본 라벨 폴더 경로  
        output_dir (str): 출력 폴더 경로
        train_ratio (float): 훈련 데이터 비율
        val_ratio (float): 검증 데이터 비율
        test_ratio (float): 테스트 데이터 비율
        seed (int): 랜덤 시드
    """
    
    print(f"\n=== 데이터셋 분할 시작 ===")
    print(f"분할 비율 - Train: {train_ratio}, Val: {val_ratio}, Test: {round(test_ratio,2)}")
    
    # 시드 설정
    random.seed(seed)
    
    # 이미지 파일 목록 가져오기
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
    image_files = []
    
    for file in os.listdir(source_images_dir):
        if any(file.lower().endswith(ext) for ext in image_extensions):
            # 대응하는 라벨 파일이 존재하는지 확인
            label_file = os.path.splitext(file)[0] + '.txt'
            label_path = os.path.join(changed_labels_dir, label_file)
            if os.path.exists(label_path):
                image_files.append(file)
            else:
                print(f"경고: {file}에 대응하는 라벨 파일이 없습니다.")
    
    print(f"총 이미지-라벨 쌍: {len(image_files)}개")
    
    if len(image_files) == 0:
        raise ValueError("유효한 이미지-라벨 쌍이 없습니다!")
    
    # 데이터 분할
    train_files, temp_files = train_test_split(image_files, test_size=(1-train_ratio), random_state=seed)
    
    if test_ratio > 0:
        val_files, test_files = train_test_split(temp_files, test_size=test_ratio/(val_ratio+test_ratio), random_state=seed)
    else:
        val_files = temp_files
        test_files = []
    
    print(f"분할 결과:")
    print(f"  Train: {len(train_files)}개")
    print(f"  Val: {len(val_files)}개")
    print(f"  Test: {len(test_files)}개")
    
    # 출력 디렉토리 생성
    splits = {
        'train': train_files,
        'val': val_files,
        'test': test_files
    }
    
    for split_name, file_list in splits.items():
        if len(file_list) == 0:
            continue
            
        # 디렉토리 생성
        img_dir = os.path.join(output_dir, 'images', split_name)
        label_dir = os.path.join(output_dir, 'labels', split_name)
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(label_dir, exist_ok=True)
        
        # 파일 복사
        for file_name in file_list:
            # 이미지 복사
            src_img = os.path.join(source_images_dir, file_name)
            dst_img = os.path.join(img_dir, file_name)
            shutil.copy2(src_img, dst_img)
            
            # 라벨 복사
            label_name = os.path.splitext(file_name)[0] + '.txt'
            src_label = os.path.join(changed_labels_dir, label_name)
            dst_label = os.path.join(label_dir, label_name)
            shutil.copy2(src_label, dst_label)
    
    print("데이터셋 분할 완료")
    return len(train_files), len(val_files), len(test_files)

def get_image_dimensions(image_path):
    """이미지 크기 가져오기"""
    
    with Image.open(image_path) as img:
        return img.width, img.height

def convert_dataset_json_to_yolo(source_images_dir, source_labels_dir, output_labels_dir):
    """
    전체 데이터셋의 JSON을 YOLO 형식으로 변환
    """
    print("=== JSON → YOLO 형식 변환 ===")
    
    class_mapping = {
        "solid_yellow_lane": 0,
        "dotted_yellow_lane": 1, 
        "double_yellow_lane": 2,
        "crosswalk": 3,
        "sidewalk": 4,
        "firehydrant": 5,
        "car": 6,
        "license_plate": 7
    }
    
    image_files = [f for f in os.listdir(source_images_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    os.makedirs(output_labels_dir, exist_ok=True)
    converted_count = 0
    
    for image_file in image_files:
        # 대응하는 JSON 파일 찾기
        json_filename = os.path.splitext(image_file)[0] + '.json'
        json_path = os.path.join(source_labels_dir, json_filename)
        
        if not os.path.exists(json_path):
            print(f"경고: {image_file}에 대응하는 JSON 파일이 없습니다.")
            continue
        
        # 이미지 크기 가져오기
        image_path = os.path.join(source_images_dir, image_file)
        try:
            width, height = get_image_dimensions(image_path)
            
            # JSON → YOLO 변환
            txt_filename = os.path.splitext(image_file)[0] + '.txt'
            txt_path = os.path.join(output_labels_dir, txt_filename)
            
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            with open(txt_path, 'w') as f:
                if 'shapes' in data:
                    for shape in data['shapes']:
                        label = shape['label']
                        
                        if label in class_mapping:
                            class_id = class_mapping[label]
                            points = shape['points']
                            
                            normalized_coords = []
                            valid_polygon = True
                            
                            for point in points:
                                x_norm = max(0.0, min(1.0, point[0] / width))
                                y_norm = max(0.0, min(1.0, point[1] / height))
                                normalized_coords.extend([x_norm, y_norm])
                            
                            # 최소 3개 점이 있는지 확인
                            if len(normalized_coords) >= 6:  # 3개 점 = 6개 좌표
                                coords_str = ' '.join(f"{coord:.6f}" for coord in normalized_coords)
                                f.write(f"{class_id} {coords_str}\n")
                            else:
                                print(f"경고: {image_file}의 {label} 객체에 충분한 점이 없습니다.")
                        else:
                            print(f"경고: {image_file}에서 알 수 없는 클래스 '{label}'를 발견했습니다.")
            
            converted_count += 1
            
        except Exception as e:
            print(f"변환 실패: {image_file} - {str(e)}")
    
    print(f"총 {converted_count}개 파일 변환 완료")
    return converted_count

def validate_polygon_labels(labels_dir, class_count=8, sample_size=None, quiet:bool=False):
    """
    폴리곤 형태 라벨 파일 검증 및 train/val/test 통합 통계
    
    Args:
        labels_dir (str): 라벨 디렉토리 경로 (output_dir/labels 경로)
        class_count (int): 예상 클래스 개수
        sample_size (int): 검사할 파일 수 (None이면 전체)
    """
    
    print(f"\n=== 폴리곤 라벨 검증 및 통합 통계 ===")
    
    # train/val/test 폴더들 확인
    splits = ['train', 'val', 'test']
    all_stats = {
        'total_valid_files': 0,
        'total_files': 0,
        'total_objects': 0,
        'total_files_with_objects': 0,
        'total_empty_files': 0,
        'total_class_distribution': {},
        'split_stats': {}
    }
    
    # 클래스 이름 정의
    class_names = [
        "solid_yellow_lane",      # 0: 노란 실선
        "dotted_yellow_lane",     # 1: 노란 점선  
        "double_yellow_lane",     # 2: 노란 이중선
        "crosswalk",              # 3: 횡단보도
        "sidewalk",               # 4: 인도
        "firehydrant",            # 5: 소화전
        "car",                    # 6: 자동차
        "license_plate"           # 7: 차량 번호판
    ]
    
    for split in splits:
        split_dir = os.path.join(labels_dir, split)
        
        if not os.path.exists(split_dir):
            print(f"경고: {split} 폴더가 존재하지 않습니다: {split_dir}")
            continue
            
        label_files = [f for f in os.listdir(split_dir) if f.endswith('.txt')]
        
        if len(label_files) == 0:
            print(f"경고: {split}에 라벨 파일이 없습니다!")
            continue
        
        # Split별 통계 초기화
        split_stats = {
            'valid_files': 0,

            
            'total_files': len(label_files),
            'total_objects': 0,
            'files_with_objects': 0,
            'empty_files': 0,
            'class_distribution': {},
            'error_files': []
        }
        
        # 전체 파일을 검사하거나 지정된 샘플 크기만큼 검사
        if sample_size is None:
            files_to_check = label_files
        else:
            files_to_check = label_files[:sample_size]
        
        print(f"{split.upper()}: {len(files_to_check)}/{len(label_files)}개 파일 검증 중...")
        
        for label_file in files_to_check:
            label_path = os.path.join(split_dir, label_file)
            
            try:
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    
                file_valid = True
                file_object_count = 0
                
                for line_num, line in enumerate(lines, 1):
                    line = line.strip()
                    if not line:  # 빈 줄 건너뛰기
                        continue
                        
                    parts = line.split()
                    if len(parts) < 7:  # 최소 class_id + 3개 점 (6개 좌표)
                        print(f"경고: {split}/{label_file}:{line_num} - 잘못된 라벨 형식 (좌표 부족)")
                        file_valid = False
                        continue
                    
                    try:
                        class_id = int(parts[0])
                    except ValueError:
                        print(f"경고: {split}/{label_file}:{line_num} - 잘못된 클래스 ID")
                        file_valid = False
                        continue
                    
                    if class_id >= class_count or class_id < 0:
                        print(f"경고: {split}/{label_file}:{line_num} - 범위를 벗어난 클래스 ID({class_id})")
                        file_valid = False
                        continue
                    
                    # 폴리곤 좌표 개수 확인 (짝수여야 함)
                    coords = parts[1:]
                    if len(coords) % 2 != 0:
                        print(f"경고: {split}/{label_file}:{line_num} - 잘못된 좌표 개수")
                        file_valid = False
                        continue
                    
                    # 좌표 값 검증
                    try:
                        coord_values = [float(c) for c in coords]
                        # 정규화된 좌표는 0-1 범위에 있어야 함
                        if any(c < 0 or c > 1 for c in coord_values):
                            print(f"경고: {split}/{label_file}:{line_num} - 좌표가 정규화 범위(0-1)를 벗어남")
                    except ValueError:
                        print(f"경고: {split}/{label_file}:{line_num} - 잘못된 좌표 값")
                        file_valid = False
                        continue
                    
                    # 클래스 분포 계산
                    split_stats['class_distribution'][class_id] = split_stats['class_distribution'].get(class_id, 0) + 1
                    all_stats['total_class_distribution'][class_id] = all_stats['total_class_distribution'].get(class_id, 0) + 1
                    split_stats['total_objects'] += 1
                    file_object_count += 1
                
                # 파일별 통계 업데이트
                if file_object_count > 0:
                    split_stats['files_with_objects'] += 1
                else:
                    split_stats['empty_files'] += 1
                
                if file_valid:
                    split_stats['valid_files'] += 1
                else:
                    split_stats['error_files'].append(label_file)
                    
            except Exception as e:
                print(f"오류: {split}/{label_file} 읽기 실패 - {e}")
                split_stats['error_files'].append(label_file)
        
        # 전체 통계에 합산
        all_stats['total_valid_files'] += split_stats['valid_files']
        all_stats['total_files'] += split_stats['total_files']
        all_stats['total_objects'] += split_stats['total_objects']
        all_stats['total_files_with_objects'] += split_stats['files_with_objects']
        all_stats['total_empty_files'] += split_stats['empty_files']
        all_stats['split_stats'][split] = split_stats
    
    # === 검증 결과 출력 ===
    print(f"\n=== 전체 데이터셋 검증 결과 ===")
    print(f"검증 완료: {all_stats['total_valid_files']}/{all_stats['total_files']} 파일 유효")
    print(f"객체가 있는 파일: {all_stats['total_files_with_objects']}개")
    print(f"빈 파일 (객체 없음): {all_stats['total_empty_files']}개")
    print(f"총 객체 수: {all_stats['total_objects']}개")
    
    # Split별 요약
    print(f"\n=== Split별 요약 ===")
    print(f"{'Split':<8} {'파일수':<8} {'객체수':<8} {'비율(%)':<8}")
    print(f"-" * 35)
    for split in splits:
        if split in all_stats['split_stats']:
            stats = all_stats['split_stats'][split]
            obj_ratio = (stats['total_objects'] / all_stats['total_objects'] * 100) if all_stats['total_objects'] > 0 else 0
            print(f"{split:<8} {stats['total_files']:<8} {stats['total_objects']:<8} {obj_ratio:<8.1f}")
    
    # === 전체 클래스별 분포 출력 ===
    print(f"\n=== 전체 데이터셋 클래스별 객체 분포 ===")
    
    if all_stats['total_class_distribution']:
        print(f"{'ID':<3} {'클래스명':<20} {'전체':<8} {'Train':<8} {'Val':<8} {'Test':<8} {'비율(%)':<8}")
        print(f"-" * 75)
        
        # 전체 클래스에 대해 출력 (ID 순서대로)
        for class_id in range(len(class_names)):
            class_name = class_names[class_id]
            total_count = all_stats['total_class_distribution'].get(class_id, 0)
            
            # Split별 개수
            train_count = all_stats['split_stats'].get('train', {}).get('class_distribution', {}).get(class_id, 0)
            val_count = all_stats['split_stats'].get('val', {}).get('class_distribution', {}).get(class_id, 0)
            test_count = all_stats['split_stats'].get('test', {}).get('class_distribution', {}).get(class_id, 0)
            
            percentage = (total_count / all_stats['total_objects'] * 100) if all_stats['total_objects'] > 0 else 0
            
            print(f"{class_id:<3} {class_name:<20} {total_count:<8} {train_count:<8} {val_count:<8} {test_count:<8} {percentage:<8.1f}")
        
        print(f"-" * 75)
        
        # 추가 통계 정보
        print(f"\n=== 추가 통계 ===")
        detected_classes = len([k for k, v in all_stats['total_class_distribution'].items() if v > 0])
        print(f"감지된 클래스 종류: {detected_classes}/{len(class_names)}개")
        
        if all_stats['total_files_with_objects'] > 0:
            print(f"파일당 평균 객체 수: {all_stats['total_objects']/all_stats['total_files_with_objects']:.1f}개 (빈 파일 제외)")
        print(f"전체 파일 기준 평균: {all_stats['total_objects']/all_stats['total_files']:.1f}개")
        
    else:
        print("클래스 분포 데이터가 없습니다.")
        for i, class_name in enumerate(class_names):
            print(f"  {i:2d} ({class_name:<18}): {0:4d}개 ({0:5.1f}%)")
            
    return all_stats['total_valid_files'] > 0

def create_dataset_yaml(dataset_path, yaml_path):
    """
    dataset.yaml 파일 생성
    """
    
    class_names = [
        'solid_yellow_lane',      # 노란 실선
        'dotted_yellow_lane',     # 노란 점선  
        'double_yellow_lane',     # 노란 이중선
        'crosswalk',              # 횡단보도
        'sidewalk',               # 인도
        'firehydrant',            # 소화전
        'car',                    # 자동차
        'license_plate'           # 차량 번호판
    ]
    
    dataset_config = {
        'path': dataset_path,
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': len(class_names),
        'names': class_names
    }
    
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(dataset_config, f, default_flow_style=False, allow_unicode=True)
    
    print(f"Dataset YAML 파일 생성: {yaml_path}")
    print(f"클래스 개수: {len(class_names)}")

def setup_training_environment():
    """
    학습 환경 설정 및 확인
    """
    print("=== 학습 환경 설정 ===")
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"사용 디바이스: {device}")
    
    if device == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    return device

def train_yolo_segmentation(
    dataset_yaml_path,
    model_name='yolov8n-seg.pt',
    # model_name='yolo11n-seg.pt',
    epochs=100,
    batch_size=16,
    img_size=640,
    project='runs/segment',
    name='road_segmentation',
    use_wandb=True
):
    """
    YOLO8 Segmentation 모델 학습 (기본값과 다른 파라미터만 명시)
    """
    
    print(f"\n=== YOLOv8 Segmentation 학습 시작 ===")
    print(f"모델: {model_name}")
    print(f"에포크: {epochs}, 배치: {batch_size}, 이미지 크기: {img_size}")
    
    # GPU 사용 설정
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"사용 디바이스: {device}")

    try:
        model = YOLO(model_name)
        print(f"✓ {model_name} 모델 로드 완료")
        
        # 학습 설정 (기본값과 다른 것만 명시)
        train_args = {
            'data': dataset_yaml_path,
            'epochs': epochs,
            'batch': batch_size,
            'imgsz': img_size,
            'project': project,
            'name': name,
            'device': device,
            'exist_ok': True,
            'seed': 42,
            'deterministic': True,
            'patience':30, 
        }
        
        if use_wandb:
            os.environ["WANDB_MODE"] = "online"
        else:
            os.environ["WANDB_MODE"] = "disabled"
        
        # 학습 실행
        results = model.train(**train_args)
        
        print("==학습 완료==")
        
        best_model_path = results.save_dir / 'weights' / 'best.pt'
        last_model_path = results.save_dir / 'weights' / 'last.pt'
        
        print(f"최고 성능 모델: {best_model_path}")
        print(f"마지막 모델: {last_model_path}")
        
        return results, str(best_model_path)
        
    except Exception as e:
        print(f"학습 중 오류 발생: {str(e)}")
        return None, None

def evaluate_model(model_path, dataset_yaml_path):
    """
    학습된 모델 성능 평가
    
    Args:
        model_path (str): 평가할 모델 경로
        dataset_yaml_path (str): 데이터셋 YAML 파일 경로
    """
    
    print(f"\n=== 모델 성능 평가 ===")
    
    try:
        model = YOLO(model_path)
        
        device = 'cuda' if torch.cuda.is_available() else 'cpu'


        # 평가 실행 (기본값과 다른 설정만 명시)
        metrics = model.val(
            data=dataset_yaml_path,
            device=device,
            save_json=True,     # 평가 결과를 coco포맷의 json파일로 저장
            save_hybrid=True    # 이미지에 정답과 예측을 모두 표시하여 시각화
        )
        
        print("평가 결과:")
        if hasattr(metrics, 'box'):
            print(f"  mAP50: {metrics.box.map50:.4f}")
            print(f"  mAP50-95: {metrics.box.map:.4f}")
            print(f"  Precision: {metrics.box.mp:.4f}")
            print(f"  Recall: {metrics.box.mr:.4f}")
        
        if hasattr(metrics, 'seg'):
            print(f"  Seg mAP50: {metrics.seg.map50:.4f}")
            print(f"  Seg mAP50-95: {metrics.seg.map:.4f}")
        
        # wandb에 결과 로그
        if wandb.run is not None:
            wandb.log({
                "eval/mAP50": metrics.box.map50 if hasattr(metrics, 'box') else 0,
                "eval/mAP50-95": metrics.box.map if hasattr(metrics, 'box') else 0,
                "eval/precision": metrics.box.mp if hasattr(metrics, 'box') else 0,
                "eval/recall": metrics.box.mr if hasattr(metrics, 'box') else 0,
                "eval/seg_mAP50": metrics.seg.map50 if hasattr(metrics, 'seg') else 0,
                "eval/seg_mAP50-95": metrics.seg.map if hasattr(metrics, 'seg') else 0,
            })
        
        return metrics
        
    except Exception as e:
        print(f"평가 중 오류 발생: {str(e)}")
        return None

def inference_example(model_path, image_path, output_dir=None):
    """
    학습된 모델로 추론 예제
    
    Args:
        model_path (str): 학습된 모델 경로
        image_path (str): 추론할 이미지 경로 (파일 또는 폴더)
        output_dir (str): 결과 저장 경로 (None이면 기본 경로 사용)
    """
    
    print(f"\n=== 추론 예제 ===")
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # 출력 디렉토리 설정
    if output_dir is None:
        output_dir = "runs/predict"  # YOLO 기본 경로
    
    # 출력 디렉토리 생성
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"모델: {model_path}")
    print(f"입력: {image_path}")
    print(f"출력 경로: {output_dir}")

    try:
        model = YOLO(model_path)
        
        # 추론 실행 - 저장 경로 지정
        results = model(
            image_path, 
            device=device, 
            save=True,           # 결과 이미지 저장
            save_txt=True,       # 텍스트 라벨 저장
            save_conf=True,      # 신뢰도 포함
            save_crop=True,      # 감지된 객체 크롭 저장 (선택적)
            project=output_dir,  # 저장 경로 지정
            name="inference",    # 실행별 폴더명
            exist_ok=True       # 기존 폴더 덮어쓰기 허용
        )
        
        # 결과 출력
        print(f"\n=== 추론 결과 ===")
        for i, r in enumerate(results):
            # 기본 정보
            boxes_count = len(r.boxes) if r.boxes is not None else 0
            masks_count = len(r.masks) if r.masks is not None else 0
            
            print(f"이미지 {i+1}:")
            print(f"  - 감지된 객체: {boxes_count}개")
            print(f"  - 분할된 객체: {masks_count}개")
            
            # 클래스별 상세 정보
            if r.boxes is not None and len(r.boxes) > 0:
                class_names = [
                    'solid_yellow_lane', 'dotted_yellow_lane', 'double_yellow_lane',
                    'crosswalk', 'sidewalk', 'firehydrant', 'car', 'license_plate'
                ]
                
                detected_classes = {}
                for box in r.boxes:
                    class_id = int(box.cls)
                    conf = float(box.conf)
                    class_name = class_names[class_id] if class_id < len(class_names) else f"class_{class_id}"
                    
                    if class_name not in detected_classes:
                        detected_classes[class_name] = []
                    detected_classes[class_name].append(conf)
                
                print("  - 클래스별 감지 결과:")
                for class_name, confidences in detected_classes.items():
                    avg_conf = sum(confidences) / len(confidences)
                    print(f"    {class_name}: {len(confidences)}개 (평균 신뢰도: {avg_conf:.3f})")
        
        # 저장된 파일 경로 출력
        save_dir = Path(output_dir) / "inference"
        if save_dir.exists():
            print(f"\n=== 저장된 파일 ===")
            print(f"결과 저장 경로: {save_dir}")
            
            # 저장된 파일 목록 출력
            saved_files = list(save_dir.glob("*"))
            if saved_files:
                for file_path in sorted(saved_files):
                    file_size = file_path.stat().st_size / 1024  # KB 단위
                    print(f"  - {file_path.name} ({file_size:.1f} KB)")
            else:
                print("  저장된 파일이 없습니다.")
        
        return results, str(save_dir)
        
    except Exception as e:
        print(f"추론 중 오류 발생: {str(e)}")
        return None, None
    
    '''
def batch_inference(model_path, input_dir, output_dir, file_extensions=None):
    """
    폴더 내 모든 이미지에 대해 배치 추론 실행
    
    Args:
        model_path (str): 학습된 모델 경로
        input_dir (str): 입력 이미지 폴더 경로
        output_dir (str): 결과 저장 폴더 경로
        file_extensions (list): 처리할 파일 확장자 목록
    """
    
    if file_extensions is None:
        file_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
    
    print(f"\n=== 배치 추론 ===")
    
    # 입력 이미지 파일 수집
    input_files = []
    for ext in file_extensions:
        pattern = os.path.join(input_dir, f"*{ext}")
        input_files.extend(glob.glob(pattern, recursive=False))
        pattern = os.path.join(input_dir, f"*{ext.upper()}")
        input_files.extend(glob.glob(pattern, recursive=False))
    
    if not input_files:
        print(f"입력 폴더에서 이미지 파일을 찾을 수 없습니다: {input_dir}")
        return None
    
    print(f"처리할 이미지: {len(input_files)}개")
    print(f"출력 경로: {output_dir}")
    
    # 배치 추론 실행
    results, save_dir = inference_example(model_path, input_dir, output_dir)
    
    return results, save_dir
    '''

def upload_model_to_hf(model_path: str,
                       repo_id: str,
                       commit_message: str,
                       private: bool = False) -> None:
    
    token = os.getenv("HUGGINGFACE_TOKEN")  # 또는 입력받기
    if not token:
        raise ValueError("HUGGINGFACE_TOKEN이 .env 파일에 없습니다.")
    api = HfApi(token=token)

    try:
        api.repo_info(repo_id)
    except Exception:
        print('레포지토리가 존재하지 않습니다.\n')

    # 모델 파일 업로드
    api.upload_file(
        path_or_fileobj=model_path,
        repo_id=repo_id,
        path_in_repo=os.path.basename(model_path),
        commit_message=commit_message
    )
    print(f"모델이 HuggingFace Hub → {repo_id} 에 업로드되었습니다.")

In [9]:
def count_class_objects(labels_path):
    
    # JSON 파일들을 순회하면서 각 클래스별 객체 수를 카운팅하는 함수
    
    
    # 대상 클래스들 정의
    target_classes = {
        'solid_yellow_lane',
        'dotted_yellow_lane', 
        'double_yellow_lane',
        'crosswalk',
        'sidewalk',
        'firehydrant',
        'car',
        'license_plate'
    }
    
    # 클래스별 카운터 초기화
    class_counts = defaultdict(int)
    
    # 처리된 파일 수 카운터
    processed_files = 0
    
    try:
        # 디렉토리 존재 확인
        if not os.path.exists(labels_path):
            print(f"경로를 찾을 수 없습니다: {labels_path}")
            return {}
        
        # 디렉토리 내 모든 파일 순회
        for filename in os.listdir(labels_path):
            if filename.endswith('.json'):
                file_path = os.path.join(labels_path, filename)
                
                try:
                    # JSON 파일 읽기
                    with open(file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    # shapes 배열에서 객체들 확인
                    if 'shapes' in data:
                        for shape in data['shapes']:
                            if 'label' in shape:
                                label = shape['label']
                                # 대상 클래스에 포함된 경우 카운트
                                if label in target_classes:
                                    class_counts[label] += 1
                    
                    processed_files += 1
                    
                except json.JSONDecodeError as e:
                    print(f"JSON 파싱 에러 - {filename}: {e}")
                except Exception as e:
                    print(f"파일 처리 에러 - {filename}: {e}")
    
    except Exception as e:
        print(f"디렉토리 접근 에러: {e}")
        return {}
    
    # 결과 출력
    print(f"\n=== 클래스별 객체 수 카운팅 결과 ===")
    print(f"처리된 파일 수: {processed_files}")
    print(f"총 발견된 객체 수: {sum(class_counts.values())}")
    print("\n클래스별 객체 수:")
    print("-" * 30)
    
    # 모든 대상 클래스에 대해 결과 출력 (0개인 클래스도 포함)
    for class_name in sorted(target_classes):
        count = class_counts[class_name]
        print(f"{class_name:20}: {count:5d}")
    
    print('\n')
    return dict(class_counts)

def count_class_objects_detailed(labels_path):
    """
    더 자세한 정보를 제공하는 카운팅 함수
    
    Args:
        labels_path (str): JSON 파일들이 있는 디렉토리 경로
        
    Returns:
        tuple: (클래스별 카운트, 파일별 상세 정보)
    """
    
    target_classes = {
        'solid_yellow_lane',
        'dotted_yellow_lane', 
        'double_yellow_lane',
        'crosswalk',
        'sidewalk',
        'firehydrant',
        'car',
        'license_plate'
    }
    
    class_counts = defaultdict(int)
    file_details = []
    
    try:
        if not os.path.exists(labels_path):
            print(f"경로를 찾을 수 없습니다: {labels_path}")
            return {}, []
        
        for filename in os.listdir(labels_path):
            if filename.endswith('.json'):
                file_path = os.path.join(labels_path, filename)
                file_info = {'filename': filename, 'classes': defaultdict(int), 'total': 0}
                
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    if 'shapes' in data:
                        for shape in data['shapes']:
                            if 'label' in shape:
                                label = shape['label']
                                if label in target_classes:
                                    class_counts[label] += 1
                                    file_info['classes'][label] += 1
                                    file_info['total'] += 1
                    
                    if file_info['total'] > 0:  # 객체가 있는 파일만 기록
                        file_details.append(file_info)
                    
                except Exception as e:
                    print(f"파일 처리 에러 - {filename}: {e}")
    
    except Exception as e:
        print(f"디렉토리 접근 에러: {e}")
        return {}, []
    
    return dict(class_counts), file_details

In [10]:
def main():

    source_images_dir = 'C:/Users/User/Downloads/dataset/images'    # 이미지 데이터 폴더
    source_labels_dir = 'C:/Users/User/Downloads/dataset/labels'    # 라벨 데이터 폴더
    changed_labels_dir = 'C:/Users/User/Downloads/dataset/change_labels'    # json > txt 변환되어 저장될 폴더
    output_dir = 'C:/Users/User/Downloads/dataset/output'       # train/test/val 나뉘어져 저장될 폴더

    convert_dataset_json_to_yolo(source_images_dir, source_labels_dir, changed_labels_dir)

    
    # 경로 검증
    if not all(os.path.exists(p) for p in [source_images_dir, changed_labels_dir]):
        print("오류: 입력 경로가 존재하지 않습니다.")
        return
    
    # 2. wandb 설정
    use_wandb_input = input("wandb 사용하시겠습니까? (y/n) [기본값: y]: ").strip().lower()
    use_wandb = use_wandb_input != 'n'
    
    wandb_initialized = False
    if use_wandb:
        project_name = 'parking_gaurd'
        run_name = input("실행 이름 (선택사항): ").strip()
        run_name = run_name if run_name else None
        
        wandb_initialized = setup_wandb(project_name, run_name)
    
    results = count_class_objects(source_labels_dir)

    # 3. 학습 환경 설정
    setup_training_environment()
    
    # 4. 데이터셋 분할
    train_ratio = float(input("훈련 데이터 비율 [기본값: 0.7]: ").strip() or "0.7")
    val_ratio = float(input("검증 데이터 비율 [기본값: 0.2]: ").strip() or "0.2")
    test_ratio = 1.0 - train_ratio - val_ratio
    
    train_count, val_count, test_count = split_dataset(
        source_images_dir, changed_labels_dir, output_dir, 
        train_ratio, val_ratio, test_ratio
    )
    
    # 5. 라벨 검증
    train_labels_dir = os.path.join(output_dir, 'labels')
    
    if not validate_polygon_labels(train_labels_dir, sample_size=None):
        print("라벨 검증에 실패했습니다.")
        return
    
    # 6. YAML 파일 생성
    yaml_path = 'output_dir/dataset.yaml'
    create_dataset_yaml(output_dir, yaml_path)
    
    # 7. 학습 설정
    print(f"\n=== 학습 설정 ===")
    model_size = input("모델 크기 (n/s/m/l/x) [기본값: n]: ").strip().lower()
    if model_size not in ['n', 's', 'm', 'l', 'x']:
        model_size = 'n'
    
    model_name = f'yolov8{model_size}-seg.pt'
    
    epochs = input("에포크 수 [기본값: 100]: ").strip()
    epochs = int(epochs) if epochs.isdigit() else 100
    
    batch_size = input("배치 크기 [기본값: 16]: ").strip()
    batch_size = int(batch_size) if batch_size.isdigit() else 16
    
    # wandb에 설정 로그
    if wandb_initialized:
        wandb.config.update({
            "epochs": epochs,
            "batch_size": batch_size,
            "model_size": model_size,
            "train_count": train_count,
            "val_count": val_count,
            "test_count": test_count,
            "train_ratio": train_ratio,
            "val_ratio": val_ratio,
            "test_ratio": test_ratio
        })
    
    # 8. 모델 학습
    print(f"\n학습을 시작합니다...")
    results, best_model_path = train_yolo_segmentation(
        dataset_yaml_path=yaml_path,
        model_name=model_name,
        epochs=epochs,
        batch_size=batch_size,
        use_wandb=wandb_initialized
    )
    
    if results is None:
        print("학습에 실패했습니다.")
        if wandb_initialized:
            wandb.finish()
        return
    
    # 9. 모델 평가
    print(f"\n모델 평가를 시작합니다...")
    metrics = evaluate_model(best_model_path, yaml_path)
    
    # wandb 종료
    if wandb_initialized:
        wandb.finish()
        print("wandb 세션 종료")
    
    print("\n=== 프로그램 완료 ===")

In [11]:
main()

=== JSON → YOLO 형식 변환 ===
총 2545개 파일 변환 완료


wandb 초기화 완료

=== 클래스별 객체 수 카운팅 결과 ===
처리된 파일 수: 2545
총 발견된 객체 수: 6169

클래스별 객체 수:
------------------------------
car                 :   728
crosswalk           :   368
dotted_yellow_lane  :   535
double_yellow_lane  :   391
firehydrant         :    49
license_plate       :   716
sidewalk            :  1566
solid_yellow_lane   :  1816


=== 학습 환경 설정 ===
사용 디바이스: cuda
GPU: NVIDIA GeForce RTX 3060
GPU 메모리: 12.0 GB

=== 데이터셋 분할 시작 ===
분할 비율 - Train: 0.7, Val: 0.2, Test: 0.1
총 이미지-라벨 쌍: 2545개
분할 결과:
  Train: 1781개
  Val: 509개
  Test: 255개
데이터셋 분할 완료

=== 폴리곤 라벨 검증 및 통합 통계 ===
TRAIN: 1781/1781개 파일 검증 중...
VAL: 509/509개 파일 검증 중...
TEST: 255/255개 파일 검증 중...

=== 전체 데이터셋 검증 결과 ===
검증 완료: 2545/2545 파일 유효
객체가 있는 파일: 2545개
빈 파일 (객체 없음): 0개
총 객체 수: 6169개

=== Split별 요약 ===
Split    파일수      객체수      비율(%)   
-----------------------------------
train    1781     4324     70.1    
val      509      1214     19.7    
test     255      631      10.2    

=== 전체 데이터셋 클래스별 객체 분포 ===
ID  클래스명           

100%|██████████| 137M/137M [00:03<00:00, 43.4MB/s] 


✓ yolov8x-seg.pt 모델 로드 완료
Ultralytics 8.3.152  Python-3.10.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=output_dir/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=road_segmentation, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=30, persp

train: Scanning C:\Users\User\Downloads\dataset\output\labels\train.cache... 1781 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1781/1781 [00:00<?, ?it/s]


...

                   all        509       1214      0.875      0.837      0.891      0.795      0.887      0.849      0.901      0.722
     solid_yellow_lane        334        365       0.89      0.833      0.923      0.786      0.904      0.848      0.938      0.661
    dotted_yellow_lane        111        114      0.721      0.632      0.701      0.566      0.769      0.675      0.726      0.482
    double_yellow_lane         68         78      0.959      0.893      0.954      0.862      0.986      0.921      0.976      0.831
             crosswalk         67         70      0.868      0.659      0.806      0.679      0.887      0.675      0.824      0.624
              sidewalk        294        311       0.95      0.912      0.963      0.887       0.95      0.912      0.964      0.889
           firehydrant          7          7      0.886      0.857      0.875      0.862      0.884      0.857      0.875      0.748
                   car        133        135      0.871      0.953   

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


평가 결과:
  mAP50: 0.8914
  mAP50-95: 0.7955
  Precision: 0.8748
  Recall: 0.8368
  Seg mAP50: 0.9008
  Seg mAP50-95: 0.7217


eval/mAP50,▁
eval/mAP50-95,▁
eval/precision,▁
eval/recall,▁
eval/seg_mAP50,▁
eval/seg_mAP50-95,▁
eval/mAP50,0.89143
eval/mAP50-95,0.79545
eval/precision,0.87483
eval/recall,0.83676
eval/seg_mAP50,0.90076


wandb 세션 종료

=== 프로그램 완료 ===


In [55]:
test_image = 'C:/Users/User/Downloads/dataset/images/IMG_8788.jpg'
if test_image and os.path.exists(test_image):
    inference_example('runs/segment/road_segmentation/weights/best.pt', test_image, 'model_results')


=== 추론 예제 ===
모델: runs/segment/road_segmentation/weights/best.pt
입력: C:/Users/User/Downloads/dataset/images/IMG_8788.jpg
출력 경로: model_results

image 1/1 C:\Users\User\Downloads\dataset\images\IMG_8788.jpg: 480x640 1 dotted_yellow_lane, 1 sidewalk, 36.9ms
Speed: 1.8ms preprocess, 36.9ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)
Results saved to model_results\inference
10 labels saved to model_results\inference\labels

=== 추론 결과 ===
이미지 1:
  - 감지된 객체: 2개
  - 분할된 객체: 2개
  - 클래스별 감지 결과:
    sidewalk: 1개 (평균 신뢰도: 0.946)
    dotted_yellow_lane: 1개 (평균 신뢰도: 0.943)

=== 저장된 파일 ===
결과 저장 경로: model_results\inference
  - crops (4.0 KB)
  - IMG_8788.jpg (2448.2 KB)
  - KakaoTalk_20250611_170805339.jpg (76.5 KB)
  - KakaoTalk_20250611_170847148.jpg (242.6 KB)
  - labels (4.0 KB)


In [ ]:
# huggingface pt 파일 업로드
repo_id  = 'won3956/parking_gaurd'
private  = False

best_model_path = 'runs/segment/road_segmentation/weights/best.pt'
upload_model_to_hf(best_model_path, repo_id, commit_message='test_upload')

best.pt: 100%|██████████| 54.8M/54.8M [00:04<00:00, 13.4MB/s]


모델이 HuggingFace Hub → won3956/parking_gaurd 에 업로드되었습니다.
